# Modèle de Classification des Incidents

---

## Objectif de ce notebook

Classifier automatiquement la **gravité d'un incident** dès sa déclaration :  
Faible / Modéré / Élevé / Critique — pour prioriser les interventions et alerter  
les bons responsables sans délai.

**Type d'apprentissage :** Supervisé (classification multi-classes à 4 classes)  
**Algorithmes comparés :** Random Forest vs SVM  
**Variable cible :** `gravite` (4 classes)  
**Déséquilibre des classes :** Faible 52,0% · Modéré 27,6% · Élevé 17,1% · Critique 3,3%

---

## 0. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Dossiers racine pour les figures et modèles
FIGURES_DIR = os.path.join('..', 'figures')
MODELS_DIR = os.path.join('..', 'models')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

if not hasattr(plt.savefig, '_is_patched'):
    _original_savefig = plt.savefig
    def savefig_root(path, *args, **kwargs):
        if isinstance(path, str) and path.startswith('figures/'):
            path = os.path.join('..', path)
        return _original_savefig(path, *args, **kwargs)
    savefig_root._is_patched = True
    plt.savefig = savefig_root

print('Imports OK ✅')

## 1. Chargement des Données

In [ ]:
df = pd.read_csv('../data/incidents_pannes.csv')
df['date_incident'] = pd.to_datetime(df['date_incident'])
df['heure_int'] = df['heure_incident'].str.split(':').str[0].astype(int)
df['mois'] = df['date_incident'].dt.month
df['annee'] = df['date_incident'].dt.year

print(f"Dataset : {df.shape[0]} incidents × {df.shape[1]} colonnes")
print()
print('=== Distribution des classes (variable cible) ===')
for g, n in df['gravite'].value_counts().items():
    print(f'  {g:<12} : {n:>4} ({n/len(df)*100:.1f}%)')
print()
print('⚠️  Déséquilibre des classes détecté — stratégie de rééquilibrage requise')

## 2. Préparation des Données

### 2.1 Sélection et justification des variables d'entrée

Variables **retenues** — disponibles au moment de la déclaration d'un incident :
- `type_incident` : nature de l'incident — forte information sur la gravité potentielle
- `depot_id` : certains dépôts sont plus risqués (taille, produits stockés)
- `produit_concerne_id` : les produits inflammables ou toxiques ont des gravités plus élevées
- `duree_arret_heures` : durée d'immobilisation — fortement corrélée à la gravité (r=0.824)
- `quantite_perdue` : volume de produit perdu — corrélé à la gravité (r=0.719)
- `heure_int` : heure de l'incident — les incidents nocturnes sont souvent plus graves
- `mois` : saisonnalité potentielle

Variables **exclues** :
- `cout_incident_usd` : connu seulement APRÈS résolution → **data leakage** si inclus
- `description`, `mesures_correctives` : texte libre difficile à encoder sans NLP
- `statut`, `date_resolution` : connus seulement après résolution → data leakage
- `operateur_responsable` : identifiant sans valeur prédictive directe

> ⚠️ **À vérifier avant mise en production :** `duree_arret_heures` et `quantite_perdue` sont très fortement corrélées à la gravité (r=0.824 et r=0.719). Il faut confirmer avec l'équipe métier / la source des données qu'elles sont **réellement connues au moment de la déclaration** de l'incident, et non calculées après coup (comme `cout_incident_usd` ou `date_resolution`, exclues ci-dessus pour la même raison). Si ce n'est pas le cas, il y a fuite de données (data leakage) et le modèle sera inutilisable au moment réel de la décision — il faudrait alors le ré-entraîner sans ces deux variables.

In [ ]:
# Variables catégorielles à encoder
CAT_FEATURES = ['type_incident', 'depot_id', 'produit_concerne_id']
NUM_FEATURES = ['duree_arret_heures', 'quantite_perdue', 'heure_int', 'mois']
FEATURES = CAT_FEATURES + NUM_FEATURES
TARGET = 'gravite'

print('Variables catégorielles :', CAT_FEATURES)
print('Variables numériques    :', NUM_FEATURES)
print(f'Total features          : {len(FEATURES)}')

### 2.2 Encodage des variables catégorielles

On utilise l'**encodage ordinal** (OrdinalEncoder) pour les variables catégorielles nominales.  
Choix justifié : Random Forest et SVM gèrent bien les entiers ordinaux.  
L'encodeur est sauvegardé avec le modèle — indispensable pour l'API.

In [ ]:
# Encodage des variables catégorielles
encoder_cat = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_encoded = df[FEATURES].copy()
df_encoded[CAT_FEATURES] = encoder_cat.fit_transform(df[CAT_FEATURES])

# Encodage de la variable cible
ORDRE_GRAVITE = ['Faible', 'Modéré', 'Élevé', 'Critique']
encoder_target = LabelEncoder()
encoder_target.classes_ = np.array(ORDRE_GRAVITE)
y = df[TARGET].map({g: i for i, g in enumerate(ORDRE_GRAVITE)}).values

X = df_encoded.values

print('Encodage appliqué ✅')
print(f'\nMapping gravité : {dict(zip(ORDRE_GRAVITE, range(4)))}')
print(f'\nShape X : {X.shape}')
print(f'Shape y : {y.shape}')
print(f'\nVérification valeurs manquantes après encodage : {np.isnan(X).sum()}')

### 2.3 Stratégie face au déséquilibre des classes

Le déséquilibre est sévère : 52% Faible vs 3,3% Critique.  
Stratégie retenue : **`class_weight='balanced'`** dans les deux algorithmes.  

Cette option pondère automatiquement les classes inversement à leur fréquence :  
un incident Critique pèse 15× plus qu'un incident Faible dans la fonction de perte.  
Avantage : pas besoin de sur-échantillonner (SMOTE), simple et efficace sur des datasets moyens.

In [ ]:
# Visualisation du déséquilibre
counts = pd.Series(y).map(dict(enumerate(ORDRE_GRAVITE))).value_counts().reindex(ORDRE_GRAVITE)
colors_g = ['#4CAF50', '#FFC107', '#FF5722', '#B71C1C']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(ORDRE_GRAVITE, counts.values, color=colors_g, edgecolor='white')
for i, (g, n) in enumerate(zip(ORDRE_GRAVITE, counts.values)):
    axes[0].text(i, n + 2, f'{n}\n({n/len(df)*100:.1f}%)',
                 ha='center', fontsize=10)
axes[0].set_title('Distribution des classes — Déséquilibre visible')
axes[0].set_ylabel("Nombre d'incidents")

# Poids calculés par class_weight='balanced'
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight('balanced', classes=np.array([0,1,2,3]), y=y)
axes[1].bar(ORDRE_GRAVITE, weights, color=colors_g, edgecolor='white')
for i, (g, w) in enumerate(zip(ORDRE_GRAVITE, weights)):
    axes[1].text(i, w + 0.1, f'{w:.2f}×', ha='center', fontsize=10)
axes[1].set_title('Poids automatiques — class_weight="balanced"')
axes[1].set_ylabel('Poids attribué')

plt.tight_layout()
plt.savefig('figures/01_desequilibre_classes.png', dpi=150)
plt.show()

### 2.4 Séparation Train / Test

On utilise `stratify=y` pour préserver la proportion de chaque classe dans les deux ensembles —  
indispensable avec des classes aussi déséquilibrées (sinon Critique pourrait être absent du test).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y   # Préserver la proportion des 4 classes
)

print(f"Train : {len(X_train)} observations")
print(f"Test  : {len(X_test)}  observations")
print()
print('Distribution dans le train :')
for i, g in enumerate(ORDRE_GRAVITE):
    n = (y_train == i).sum()
    print(f'  {g:<12} : {n:>3} ({n/len(y_train)*100:.1f}%)')
print()
print('Distribution dans le test :')
for i, g in enumerate(ORDRE_GRAVITE):
    n = (y_test == i).sum()
    print(f'  {g:<12} : {n:>3} ({n/len(y_test)*100:.1f}%)')

## 3. Algorithme 1 — Random Forest Classification

Random Forest construit un ensemble d'arbres de décision en parallèle (bagging).  
La prédiction finale est la classe majoritaire parmi tous les arbres.  
Avec `class_weight='balanced'`, les incidents Critiques pèsent ~15× plus.

In [ ]:
model_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=2,
    class_weight='balanced',  # Gestion du déséquilibre
    random_state=42,
    n_jobs=-1
)
model_rf.fit(X_train, y_train)
print('Random Forest entraîné ✅')

y_pred_rf = model_rf.predict(X_test)

print('\n=== Métriques Random Forest ===')
print(classification_report(y_test, y_pred_rf,
                             target_names=ORDRE_GRAVITE))

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf  = f1_score(y_test, y_pred_rf, average='macro')
print(f'Accuracy     : {acc_rf:.3f}')
print(f'F1 macro     : {f1_rf:.3f}')

In [ ]:
# Matrice de confusion Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=ORDRE_GRAVITE,
            yticklabels=ORDRE_GRAVITE, ax=axes[0])
axes[0].set_title('Matrice de confusion — Random Forest')
axes[0].set_xlabel('Prédit')
axes[0].set_ylabel('Réel')

# Importance des variables
importance_rf = pd.Series(
    model_rf.feature_importances_, index=FEATURES
).sort_values(ascending=True)
importance_rf.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Importance des variables — Random Forest')
axes[1].set_xlabel('Importance (Gini)')

plt.tight_layout()
plt.savefig('figures/02_rf_confusion_importance.png', dpi=150)
plt.show()

print('\nTop 3 variables les plus importantes :')
for feat, imp in importance_rf.tail(3).iloc[::-1].items():
    print(f'  {feat:<30} : {imp:.4f}')

## 4. Algorithme 2 — SVM (Support Vector Machine)

SVM cherche l'hyperplan qui maximise la marge entre les classes.  
Avec un noyau RBF, il peut capturer des séparations non linéaires entre les gravités.  
`class_weight='balanced'` applique la même pondération que pour Random Forest.

In [ ]:
from sklearn.preprocessing import StandardScaler as SC

# SVM est sensible aux échelles — normalisation obligatoire
scaler_svm = SC()
X_train_s = scaler_svm.fit_transform(X_train)
X_test_s  = scaler_svm.transform(X_test)

model_svm = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    class_weight='balanced',
    random_state=42,
    probability=True   # Pour obtenir des probabilités par classe
)
model_svm.fit(X_train_s, y_train)
print('SVM entraîné ✅')

y_pred_svm = model_svm.predict(X_test_s)

print('\n=== Métriques SVM ===')
print(classification_report(y_test, y_pred_svm,
                             target_names=ORDRE_GRAVITE))

acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm  = f1_score(y_test, y_pred_svm, average='macro')
print(f'Accuracy  : {acc_svm:.3f}')
print(f'F1 macro  : {f1_svm:.3f}')

In [ ]:
# Matrice de confusion SVM
cm_svm = confusion_matrix(y_test, y_pred_svm)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=ORDRE_GRAVITE,
            yticklabels=ORDRE_GRAVITE, ax=ax)
ax.set_title('Matrice de confusion — SVM')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.savefig('figures/03_svm_confusion.png', dpi=150)
plt.show()

## 5. Comparaison et Sélection du Meilleur Modèle

In [ ]:
# Tableau comparatif
resultats = pd.DataFrame({
    'Modèle'    : ['Random Forest', 'SVM (RBF)'],
    'Accuracy'  : [acc_rf,  acc_svm],
    'F1 macro'  : [f1_rf,   f1_svm],
})
print(resultats.to_string(index=False))

# F1 par classe
print('\nF1-score par classe :')
f1_rf_classes  = f1_score(y_test, y_pred_rf,  average=None)
f1_svm_classes = f1_score(y_test, y_pred_svm, average=None)
for i, g in enumerate(ORDRE_GRAVITE):
    print(f'  {g:<12} : RF={f1_rf_classes[i]:.3f} · SVM={f1_svm_classes[i]:.3f}')

In [ ]:
# Barplot comparatif F1 par classe
x = np.arange(len(ORDRE_GRAVITE))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 par classe
axes[0].bar(x - width/2, f1_rf_classes,  width, label='Random Forest',
             color='steelblue', edgecolor='white')
axes[0].bar(x + width/2, f1_svm_classes, width, label='SVM',
             color='darkorange', edgecolor='white')
axes[0].set_title('F1-score par classe de gravité')
axes[0].set_ylabel('F1-score')
axes[0].set_xticks(x)
axes[0].set_xticklabels(ORDRE_GRAVITE)
axes[0].set_ylim(0, 1.1)
axes[0].legend()
axes[0].axhline(0.5, color='red', linestyle=':', linewidth=1, alpha=0.5)

# Accuracy et F1 macro
metriques = ['Accuracy', 'F1 macro']
vals_rf  = [acc_rf,  f1_rf]
vals_svm = [acc_svm, f1_svm]
x2 = np.arange(len(metriques))
axes[1].bar(x2 - width/2, vals_rf,  width, label='Random Forest',
             color='steelblue', edgecolor='white')
axes[1].bar(x2 + width/2, vals_svm, width, label='SVM',
             color='darkorange', edgecolor='white')
for i, (vrf, vsvm) in enumerate(zip(vals_rf, vals_svm)):
    axes[1].text(i - width/2, vrf  + 0.01, f'{vrf:.3f}',  ha='center', fontsize=9)
    axes[1].text(i + width/2, vsvm + 0.01, f'{vsvm:.3f}', ha='center', fontsize=9)
axes[1].set_title('Accuracy et F1 macro global')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(metriques)
axes[1].set_ylim(0, 1.1)
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/04_comparaison_modeles.png', dpi=150)
plt.show()

In [ ]:
# Analyse des erreurs — quelles classes sont confondues ?
# Focus sur le modèle Random Forest
erreurs_rf = pd.DataFrame({
    'Réel'  : [ORDRE_GRAVITE[i] for i in y_test],
    'Prédit': [ORDRE_GRAVITE[i] for i in y_pred_rf]
})
erreurs_rf = erreurs_rf[erreurs_rf['Réel'] != erreurs_rf['Prédit']]

print(f'Nombre total d\'erreurs RF : {len(erreurs_rf)}')
print()
print('Confusions les plus fréquentes (Réel → Prédit) :')
confusion_paires = erreurs_rf.groupby(['Réel','Prédit']).size().sort_values(ascending=False)
print(confusion_paires.head(8))

print()
print('=== Analyse métier des erreurs ===')
critique_manque = erreurs_rf[erreurs_rf['Réel'] == 'Critique']
print(f'Incidents Critiques mal classifiés : {len(critique_manque)}')
if len(critique_manque) > 0:
    print('  Prédit comme :')
    print(critique_manque['Prédit'].value_counts())
    print('  ⚠️  Classer un Critique comme Faible/Modéré est la pire erreur possible')

**📝 Sélection et Justification :**

> **Métrique prioritaire : F1-score macro** — pénalise équitablement les erreurs sur toutes les classes, y compris les classes minoritaires (Critique). L'accuracy seule serait trompeuse : un modèle qui prédirait toujours "Faible" obtiendrait 52% d'accuracy sans jamais détecter les incidents graves.
>
> **Analyse des erreurs de classification :** les confusions les plus fréquentes se produisent entre classes adjacentes (Faible ↔ Modéré, Modéré ↔ Élevé) — ce qui est acceptable d'un point de vue métier. La confusion Critique → Faible est la plus grave : elle signifie que l'équipe de réponse ne sera pas alertée alors que la situation est critique. Le modèle doit minimiser cette erreur spécifique.
>
> **Modèle retenu : Random Forest** car il offre en plus du meilleur F1-score :
> - L'importance des variables — cruciale pour expliquer les prédictions au directeur
> - Des probabilités par classe (`predict_proba`) permettant d'afficher un score de confiance
> - Une meilleure interprétabilité que SVM
> - Pas besoin de normalisation supplémentaire pour l'API (contrairement à SVM)

## 6. Sauvegarde du Modèle et des Encodeurs

In [ ]:
# Sauvegarder le modèle Random Forest
with open(os.path.join(MODELS_DIR, 'model_incidents.pkl'), 'wb') as f:
    pickle.dump(model_rf, f)

# Sauvegarder les encodeurs (INDISPENSABLES pour l'API)
encoders = {
    'encoder_cat'    : encoder_cat,
    'cat_features'   : CAT_FEATURES,
    'num_features'   : NUM_FEATURES,
    'all_features'   : FEATURES,
    'ordre_gravite'  : ORDRE_GRAVITE,
}
with open(os.path.join(MODELS_DIR, 'encoders_incidents.pkl'), 'wb') as f:
    pickle.dump(encoders, f)

print(f"Modèle sauvegardé   : {os.path.join(MODELS_DIR, 'model_incidents.pkl')} ✅")
print(f"Encodeurs sauvegardés : {os.path.join(MODELS_DIR, 'encoders_incidents.pkl')} ✅")

## 7. Conclusions

---

### 🔍 Résultats clés

> 1. **Déséquilibre sévère** (52% Faible vs 3,3% Critique) traité par `class_weight='balanced'` — solution simple et efficace sans sur-échantillonnage.
> 2. La **durée d'arrêt** (`duree_arret_heures`) est la variable la plus prédictive de la gravité (r=0.824) — confirmé par l'importance des variables.
> 3. Les confusions se produisent entre **classes adjacentes** — comportement normal et acceptable métier.
> 4. Classer un incident **Critique comme Faible est la pire erreur** — à surveiller en production.
> 5. **Random Forest retenu** pour son F1 macro supérieur, son interprétabilité et sa compatibilité directe avec l'API.
> 6. Fichiers à transmettre à SEGNEDJI : `model_incidents.pkl` + `encoders_incidents.pkl`.

---

### ✅ Décisions

> - **Modèle retenu** : Random Forest (`class_weight='balanced'`, `n_estimators=300`)
> - **Fichiers sauvegardés** : `models/model_incidents.pkl` + `models/encoders_incidents.pkl`
> - **Règle d'escalade** : `gravite >= Élevé` → notification immédiate (implémente `declencherEscalade()` de la classe `Incident`)
> - **Métrique principale** : F1-score macro (pas l'accuracy)
> - **Score de confiance** : `predict_proba` exposé dans l'API pour afficher la probabilité sur le dashboard

---

### 📁 Figures produites

| Fichier | Description |
|---|---|
| `01_desequilibre_classes.png` | Distribution des classes + poids automatiques |
| `02_rf_confusion_importance.png` | Matrice de confusion RF + importance des variables |
| `03_svm_confusion.png` | Matrice de confusion SVM |
| `04_comparaison_modeles.png` | F1 par classe + Accuracy et F1 macro global |